## 공공데이터 포털 (민간 아파트 가격 동향)
- 파일명: house_price_preprocessing_2021.csv

In [ ]:
# 한글 설치후 런타인 다시 시작
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

### Part1. 데이터 가져오기 및 확인

In [ ]:
# 모듈 연결하기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
## 구글 파일 업로드
from google.colab import files
files.upload()

In [ ]:
# 데이터 읽어오기
df = pd.read_csv("house_price_preprocessing_2021.csv", encoding='cp949')

# 데이터 확인하기
df.tail()

In [ ]:
# 열이름은 "분양가격" 으로 변경하기
df = df.rename(columns={df.columns[-1]: '분양가격'})
df.head(3)

In [ ]:
# 데이터 전체 상태 확인하기
df.info()

In [ ]:
# 데이터 타입만 확인하기
df.dtypes

In [ ]:
# 결측지 여부 확인
df.isna().sum()

## Part2. 데이터 전처리

분양가격 컬럼(열)의 데이터 타입을 int로 바꿈

In [ ]:
# 데이터에 잘못 입력된 데이터 0으로 변경하기
df.loc[df['분양가격'] == '  ', "분양가격"] = "0"

In [ ]:
# NaN 값 확인
df[df['분양가격'].isna()]

In [ ]:
# 데이터에서 결측지 값을 0 으로 변경하기
df['분양가격'] = df['분양가격'].fillna('0')
df['분양가격'].isna().sum()

In [ ]:
# 데이터에서 "," 제거하기(예: "2,222" => "2222"
df['분양가격'] = df['분양가격'].str.replace(',', '')

In [ ]:
# 데이터에서 "-" 기호 제거하기
df['분양가격'] = df['분양가격'].str.replace('-', '0')

In [ ]:
# loc로 10개 데이터 출력하기
df.loc[0:10]

In [ ]:
# 분양가격 데이터 int로 수정하기
df['분양가격'].astype(int)

In [ ]:
df['분양가격'] = df['분양가격'].astype(float)  # df.astype('분양가격':int})
df.dtypes

In [ ]:
# 데이터 정보 확인하기
df.info()

In [ ]:
# '분양가격' 열에서 값이 0인 것들을 모두 출력
df.loc[ df['분양가격'] == 0]

In [ ]:

# 분양가격이 0인 행의 index 모두 가져와 idx에 저장하기
idx = df.loc[ df['분양가격'] == 0].index
idx

In [ ]:
# idx 행들을 drop, 행은 axis=0 (열은 axis=1)
dfb=df.copy()   # 기존 데이터 dfb 복사
df = df.drop(idx, axis=0)

# count()함수로 데이터 갯수 확인하기
df.count()

In [ ]:
# numpy의 브로드 캐스트 개념 이용(모든 열 한꺼번에 사칙연산 가능)
# 분양가격/3.3 결과 다시 분양가격에 업데이트
df['분양가격'] = df['분양가격']/3.3

# 열 이름을 '분양가격'을 '평당분양가격'으로 변경
df = df.rename(columns={'분양가격': '평당분양가격'})

In [ ]:
# 데이터 5개 출력하고 확인하기
df.head(10)

In [ ]:
# 규모구분 " "로 분류하기(구분/면적)
# 구분: 전용면적/모든면적
# 면적: 모든면적/60제곱미터이하/85제곱미터이하/102제곱미터이하/102제곱미터초과

area = []  # 구분
gu = []    # 면적

for f in df['규모구분']:
    if len(f.split()) == 1:
        area.append(f.split()[0])
        gu.append("모든면적")
    elif len(f.split()) == 2:
        area.append(f.split()[0])
        gu.append(f.split()[1])
    else:
        area.append(f.split()[0])
        gu.append(f.split()[2])

df['구분']=area
df['면적']=gu

df.head()

In [ ]:
# 규모구분 " "로 분류하기(구분/면적)
# 구분: 전용면적/모든면적
# 면적: 모든면적/60제곱미터이하/85제곱미터이하/102제곱미터이하/102제곱미터초과

area = []  # 구분
gu = []    # 면적

for f in df['규모구분']:
    area.append(f.split()[0])
    gu.append(f.split()[-1])

df['구분']=area
df['면적']=gu

df.head()

In [ ]:
# df에서 '규모구분' 필드 제거하기
del df['규모구분']

df.head()

## Part3.데이터 시각화

In [ ]:
# 시스템에 등록된 폰트 확인
import matplotlib.font_manager as fm

sys_font=fm.findSystemFonts()
print(f"sys_font number: {len(sys_font)}")
sys_font

In [ ]:
# 그래프 출력 시 이상한 에러들 무시
import warnings
warnings.filterwarnings("ignore")

# 그래프 그릴 때 한글 깨짐 방지 설정
import os

#Mac OS의 경우와 그 외 OS의 경우로 나누어 설정
#if os.name == 'posix':
#    plt.rc("font", family="NanumGothic.ttf")
#elif os.name =='Windows':
#    plt.rc("font", family="Malgun Gothic.ttf")
#else:
#    plt.rc("font", family="NanumGothic")

plt.rc("font", family="NanumGothic")

In [ ]:
# '지역명'으로 묶어서 '분양가격'의 평균 출력하기
df_group=df.groupby('지역명')['평당분양가격'].mean()
df_group

In [ ]:
df_group.plot(kind="bar", figsize=(10, 6))
plt.show()

In [ ]:
# seaborn 패키지로 막대 그래프 그리기
plt.figure(figsize=(10, 6))
sns.barplot(x='지역명', y='평당분양가격', data=df[df['구분']!='모든면적'])
plt.show()

In [ ]:
# 연도별 분양가격 bar 차트
plt.figure(figsize=(10, 6))
sns.barplot(x='연도', y='평당분양가격', data=df[df['구분']!='모든면적'])
plt.show()

In [ ]:
# 연도별 서울의 분양가격 bar 차트
plt.figure(figsize=(10, 6))
sns.barplot(x='연도', y='평당분양가격', data=df[(df['구분']!='모든면적')&(df['지역명']=='서울')])
plt.show()

피벗으로 분석

In [ ]:
# pivot_table로 데이터 분석(기본적 평균값 출력)
df_pivot=pd.pivot_table(df[df['구분']=='모든면적'], index='지역명', columns='연도', values='평당분양가격')
df_pivot

In [ ]:
# pivot_table로 데이터 분석(기본적 평균값 출력)
df_pivot=pd.pivot_table(df[df['구분']!='모든면적'], index='지역명', columns='연도', values='평당분양가격')
df_pivot

- np.sum - 합계 / np.mean - 평균 / np.median - 중위값 / np.var - 분산
- np.std - 표준편차 / np.min - 최소값 / np.max - 최대값

In [ ]:
# 서울 연도별 평균분양가격 출력
df_pivot.loc['서울']

In [ ]:
# 서울의 년도별 평당분양가격 꺾은선 차트 출력
plt.style.use(["ggplot"])
plt.figure(figsize=(12, 6))
df_pivot.loc['서울'].plot()
plt.show()

In [ ]:
# aggfunc 옵션에 np.max를 입력하여 최대값 데이터를 출력
df_pivot_max=pd.pivot_table(df[df['구분']=='모든면적'], index='지역명', columns='연도',
                            values='평당분양가격', aggfunc=np.max)
df_pivot_max

### 면적에 따른 지역별/년도별 분양 건수 확인

In [ ]:
# groupby()를 이용해 '지역명'으로 묶어 count 구하기
df_cnt=df[df['구분']!='모든면적'].groupby(['지역명','연도','면적'])[['평당분양가격']].count()
df_cnt

In [ ]:
# index 재설정하기
df_cnt=df_cnt.reset_index(drop=False)
df_cnt

In [ ]:
# 데이터 계산
df_cnt_g=df_cnt.groupby('지역명')[['평당분양가격']].sum()

# 시각화
plt.figure(figsize=(12, 5))
sns.barplot(x=df_cnt_g.index, y=df_cnt_g['평당분양가격'], data=df_cnt_g)
plt.show()


In [ ]:
# 시각화
plt.figure(figsize=(12, 5))
sns.barplot(x=df_cnt_g.index, y=df_cnt_g['평당분양가격'], data=df_cnt.groupby('지역명')[['평당분양가격']].sum())
plt.show()

### 서울의 면적별/연도별 평당 분양가격 확인

In [ ]:
df_seoul=df[(df['구분']!='모든면적') & (df['지역명']=='서울')].groupby(['지역명','연도','면적'])[['평당분양가격']].mean()
df_seoul.head()

In [ ]:
# index 재설정하기
df_seoul=df_seoul.reset_index(drop=False)
df_seoul

In [ ]:
gu_m = df_seoul['면적'].unique()
gu_m

In [ ]:
# 면적별 분양가 변동현환 차트 출력
gu_m = df_seoul['면적'].unique()

plt.style.use(["ggplot"])
plt.figure(figsize=(12, 6))

for gu_name in gu_m:
    dfs=df_seoul[df_seoul['면적']==gu_name]
    plt.plot(dfs['연도'], dfs['평당분양가격'], label=gu_name)

plt.legend()
plt.show()

In [ ]:
# 면적별 분양가 변동현환 각각 차트로 출력
gu_m = list(df_seoul['면적'].unique())

plt.style.use(["ggplot"])
plt.figure(figsize=(12, 16))

for i,val in enumerate(gu_m):
    plt.subplot(4, 1, i+1)
    dfs=df_seoul[df_seoul['면적']==val]
    plt.plot(dfs['연도'], dfs['평당분양가격'])
    plt.title(val)
plt.show()

dfb 데이터로 전처리
- 0 값 처리: 지역별/면적별 평균

In [ ]:
dfb

In [ ]:
# 규모구분을 구분/면적으로 열 생성
area = []  # 구분
gu = []    # 면적

for f in dfb['규모구분']:
    area.append(f.split()[0])
    gu.append(f.split()[-1])

dfb['구분']=area
dfb['면적']=gu

del dfb['규모구분']

dfb.head()

In [ ]:
dfb.head()

In [ ]:
# 지역별/면적별/년도별 평균 계산
dfb_g=dfb[dfb["분양가격"] != 0].groupby(['지역명','연도','면적'])[["분양가격"]].mean()
dfb_g.reset_index(drop=False, inplace=True)
dfb_g

In [ ]:
dfb[dfb['분양가격']==0]

In [ ]:
dfb_g[(dfb_g['지역명']=="강원")&(dfb_g['연도']==2015)&(dfb_g['면적']=="102제곱미터이하")]['분양가격']